# `get_df_info` — быстрый аудит колонок датафрейма

Функция-однострочник для первого взгляда на табличные данные. По любому `DataFrame` возвращает
сводную таблицу: для каждой колонки — тип, число уникальных значений, флаг `id_like`,
доли пропусков (`nans`) / нулей (`zeros`) / пустых строк (`empty`), самое частое значение (`top`)
и пару примеров.

Колонки отсортированы по `trash_score` — эвристике «мусорности»: она высока, если колонка
в основном состоит из пропусков/нулей/пустых строк либо если одно значение занимает больше
доли `thr`. Это **сигнал «посмотри внимательно», а не вердикт** — наверх всплывает то, что
стоит проверить глазами.

Удобно держать в утилитном модуле и дёргать в начале любого EDA.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# recommended: add your own rcparams here
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

In [ ]:
def get_df_info(df, thr=0.95, id_thr=0.99, max_str_len=15):
    '''
    Сводка по колонкам датафрейма в виде датафрейма, отсортированная по trash_score.

    trash_score — эвристический сигнал «потенциально проблемной» колонки:
    он высок, если колонка в основном состоит из пропусков / нулей / пустых строк
    (их суммарная доля) либо если одно значение занимает больше доли thr.

    df: исходный датафрейм
    thr: порог доминирования одного значения для trash_score (по умолчанию 0.95)
    id_thr: порог доли уникальных значений для флага id_like (по умолчанию 0.99)
    max_str_len: ограничение на длину текстовых полей при выводе (по умолчанию 15)

    returns: pd.DataFrame с инфой по колонкам
    '''
    info_dict = {}
    total_rows = len(df)

    def format_share(value):
        if value == 0:
            return '—'
        return f"{value:.3f}"

    def format_repr(value, max_len=max_str_len):
        s = str(value)
        if len(s) > max_len:
            return s[:max_len] + '...'
        return s

    for col_name in df.columns:
        col = df[col_name]

        dtype = col.dtype.name

        # один проход хешированием: даёт и nunique, и самое частое значение
        vc_all = col.value_counts(dropna=False)
        unique_count = len(vc_all)

        na_mask = vc_all.index.isna()
        nan_count = int(vc_all[na_mask].sum())
        vc_nonnull = vc_all[~na_mask]
        n_nonnull = total_rows - nan_count

        nan_share = nan_count / total_rows if total_rows else 0
        zero_share = (col == 0).mean() if pd.api.types.is_numeric_dtype(col) else 0
        empty_str_share = (col == '').mean() if hasattr(col, 'str') else 0

        if len(vc_nonnull) == 0:
            top_str = 'N/A'
            top_share = 0
            nunique_nonnull = 0
        else:
            top_share = vc_nonnull.iloc[0] / n_nonnull
            top_element = vc_nonnull.index[0]
            top_str = f"{format_share(top_share)} ('{format_repr(top_element)}')"
            nunique_nonnull = len(vc_nonnull)

        # examples: первые два уникальных в порядке появления
        unique_values = col.dropna().unique()
        if len(unique_values) == 0:
            example_1, example_2 = 'N/A', 'N/A'
        elif len(unique_values) == 1:
            example_1, example_2 = unique_values[0], 'N/A'
        else:
            example_1, example_2 = unique_values[0], unique_values[1]
        examples_str = f"'{format_repr(example_1)}', '{format_repr(example_2)}'"

        is_float = pd.api.types.is_float_dtype(col)
        id_like = 'id_like' if (not is_float and n_nonnull > 0
                                and nunique_nonnull / n_nonnull >= id_thr) else ''

        bad_values_share = nan_share + zero_share + empty_str_share
        vc_trash = top_share if top_share > thr else 0
        trash_score = max(bad_values_share, vc_trash)

        info_dict[col_name] = {
            'dtype': dtype,
            'unique': unique_count,
            'id_like': id_like,
            'nans': format_share(nan_share),
            'zeros': format_share(zero_share),
            'empty': format_share(empty_str_share),
            'top': top_str,
            'examples': examples_str,
            'trash_score': trash_score,
        }

    result_df = pd.DataFrame.from_dict(info_dict, orient='index')
    result_df.sort_values(by='trash_score', ascending=False, inplace=True)
    result_df['trash_score'] = result_df['trash_score'].apply(format_share)

    return result_df

### Демонстрация на Titanic (с искусственно внесёнными пропусками)

In [4]:
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# немного искусственных пропусков для демонстрации
rng = np.random.default_rng(0)
df.loc[rng.choice(df.index, 100, replace=False), 'Fare'] = np.nan
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,NaN,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,NaN,NaN,S


In [5]:
get_df_info(df)

,dtype,unique,id_like,nans,zeros,empty,top,examples,trash_score
Cabin,object,148,,0.771,—,—,0.020 ('G6'),"'C85', 'C123'",0.771
Parch,int64,7,,—,0.761,—,0.761 ('0'),"'0', '1'",0.761
SibSp,int64,7,,—,0.682,—,0.682 ('0'),"'1', '0'",0.682
Survived,int64,2,,—,0.616,—,0.616 ('0'),"'0', '1'",0.616
Age,float64,89,,0.199,—,—,0.042 ('24.0'),"'22.0', '38.0'",0.199
Fare,float64,243,,0.112,0.016,—,0.051 ('13.0'),"'7.25', '71.2833'",0.128
Embarked,object,4,,0.002,—,—,0.724 ('S'),"'S', 'C'",0.002
PassengerId,int64,891,id_like,—,—,—,0.001 ('1'),"'1', '2'",—
Name,object,891,id_like,—,—,—,"0.001 ('Braund, Mr. Owe...')","'Braund, Mr. Owe...', 'Cumings, Mrs. J...'",—
Pclass,int64,3,,—,—,—,0.551 ('3'),"'3', '1'",—


### Известные ограничения и как читать `trash_score`

- `trash_score` подсвечивает **потенциально** проблемные колонки, а не выносит приговор.
  В частности, осмысленные **бинарные признаки `0/1`** всплывут наверх из-за высокой доли нулей,
  а **булевы** колонки — потому что `False` считается нулём (`is_numeric_dtype(bool) == True`).
  Это ожидаемо: задача функции — собрать кандидатов на ручной просмотр, решение остаётся за тобой.
- `id_like` — тоже эвристика (доля уникальных ≥ `id_thr`), и вешается только на не-`float` колонки,
  чтобы непрерывные величины не принимались за идентификаторы. Редкие целочисленные/строковые
  колонки с почти-уникальными значениями всё равно могут попасть под флаг.
- Подсчёт уникальных требует прохода хешированием — на очень широких/больших датасетах это
  основная стоимость функции.
- «Пустоту» в реальных данных кодируют по-разному (текстовый `'NaN'`, прочерк, `0` строкой) —
  такие случаи стоит нормализовать заранее.